In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 160)
OUT = ROOT / "outputs"
def show(name): display(Image(filename=str(OUT / "figures" / f"{name}.png")))


# 01 Data acquisition and validation
Downloading is done by `scripts/download_data.py` (not re-run here). This notebook inspects the manifest, the quality report and the processed data.

In [2]:
import json
from src.data.align import load_processed
print(pd.read_csv(ROOT / "data" / "data_manifest.csv")[["file", "source", "retrieved", "frequency", "units"]].to_string())

                                           file                          source   retrieved frequency                                                        units
0     data/raw/10_Industry_Portfolios_Daily.csv  Kenneth R. French Data Library  2026-10-24     daily                 percent (not decimal); -99.99/-999 = missing
1  data/raw/F-F_Research_Data_Factors_daily.csv  Kenneth R. French Data Library  2026-10-24     daily                 percent (not decimal); -99.99/-999 = missing
2                      data/raw/VIX_History.csv        Cboe VIX historical data  2026-10-24     daily  index points; columns OPEN,HIGH,LOW,CLOSE; dates MM/DD/YYYY


In [3]:
print(pd.DataFrame(json.loads((ROOT / "data/processed/data_quality_report.json").read_text())).T.to_string())

                                        0              1           2                            3
dataset                    industry_daily  factors_daily   vix_daily                      aligned
start                          1926-07-01     1926-07-01  1990-01-02                   1990-01-02
end                            2026-08-31     2026-08-31  2026-10-23                   2026-08-31
n_rows                              26317          26317        9301                         9233
duplicate_dates                       0.0            0.0         0.0                          NaN
monotonic_index                      True           True        True                          NaN
missing_cells                         0.0            0.0         0.0                          NaN
max_calendar_gap_days                12.0           12.0         7.0                          NaN
n_abs_return_gt_threshold             0.0            0.0         NaN                          NaN
vix_missing_before_f

In [4]:
ind, aux = load_processed()
print(ind.shape, ind.index.min().date(), ind.index.max().date())
print("duplicate dates:", ind.index.duplicated().sum(), "| missing cells:", int(ind.isna().sum().sum()))
print("largest |daily return|:", float(ind.abs().max().max()))   # decimals: ~0.2 = 20%
aux.describe().T

(9233, 10) 1990-01-02 2026-08-31
duplicate dates: 0 | missing cells: 0
largest |daily return|: 0.1968


,count,mean,std,min,25%,50%,75%,max
risk_free_rate,9233.0,0.000107,0.000093,0.00,0.0000,0.0001,0.0002,0.0003
vix_close,9233.0,19.439862,7.730399,9.14,13.9900,17.6000,22.7000,82.6900
market_return,9233.0,0.000481,0.011371,-0.12,-0.0043,0.0007,0.0058,0.1136


In [5]:
gaps = ind.index.to_series().diff().dt.days
print(gaps.value_counts().sort_index().head(10))

date
1.0    7230
2.0      89
3.0    1675
4.0     235
5.0       2
7.0       1
Name: count, dtype: int64
